# Arrow IPC - JavaScript

All 10 JavaScript examples from [docs/core/ipc.md](https://platob.github.io/yggdryl/core/ipc/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install @yggdryl/node
```

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('@yggdryl/node')

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL', null], new arrow.Utf8()),
})

// The name says Arrow IPC, so no call names an encoding.
const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.arrows'))
handle.writeArrowBatchReader(table)

assert.equal(handle.readArrowField().name, 'row')
assert.equal(handle.readArrowBatchReader().toTable().numRows, 2)

fs.rmSync(root, { recursive: true, force: true })

## Reading and writing are both readers

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, IOBase, MimeType } = require('@yggdryl/node')

const batches = [0, 2, 4].map(
  (start) =>
    new arrow.Table({
      id: arrow.vectorFromArray([BigInt(start), BigInt(start + 1)], new arrow.Int64()),
    }).batches[0],
)

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM

// An Arrow JS Table, one RecordBatch, an array of them, or Arrow IPC bytes:
// `BatchReader.from` turns whatever is in hand into the shape a write takes.
handle.writeArrowBatchReader(BatchReader.from(batches))

const reader = handle.readArrowBatchReader()
// The schema is known before a single batch is decoded.
assert.deepEqual([...reader.field.dataType].map((child) => child.name), ['id'])

let rows = 0
for (const batch of reader) rows += batch.numRows
assert.equal(rows, 6)

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, IOBase, MimeType } = require('@yggdryl/node')

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM

// Apache Arrow JS owns the encoding of what a caller already holds, so the
// four batches cross the boundary once, as one Arrow IPC stream.
const produced = [0, 1, 2, 3].map(
  (start) =>
    new arrow.Table({ id: arrow.vectorFromArray([BigInt(start)], new arrow.Int64()) })
      .batches[0],
)
handle.writeArrowBatchReader(BatchReader.from(produced))

assert.equal([...handle.readArrowBatchReader()].length, 4)

## Column pushdown

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { Field, IOBase, MimeType, fields } = require('@yggdryl/node')

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
handle.writeArrowBatchReader(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
    venue: arrow.vectorFromArray(['XNAS', 'XNAS'], new arrow.Utf8()),
  }),
)

// One of the three columns, declared as this read's schema.
const wanted = fields.struct('row', [Field.from('id: int64')], { nullable: false })

const projected = handle.readArrowBatchReader(handle.recordOptions().withSchema(wanted))
assert.deepEqual([...projected.field.dataType].map((child) => child.name), ['id'])
assert.equal(projected.toTable().numCols, 1)

// The stream itself is unchanged: it still carries all three.
assert.equal(handle.readArrowField().dataType.length, 3)

## The stream carries its schema

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { IOBase, MimeType } = require('@yggdryl/node')

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
handle.writeArrowBatchReader(
  new arrow.Table({ id: arrow.vectorFromArray([7n], new arrow.Int64()) }),
)

// A reader that declares nothing recovers the schema from the bytes.
assert.equal(handle.readArrowField().name, 'row')

// Arrow names columns, not the record; the root name is chosen on this side.
const named = handle.recordOptions().withRootName('trade')
assert.equal(handle.readArrowField(named).name, 'trade')
assert.deepEqual([...handle.readArrowField().dataType].map((child) => child.name), ['id'])

## Content coding comes from the name

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('@yggdryl/node')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const written = []
for (const name of ['trades.arrows', 'trades.arrows.gz', 'trades.arrows.zst']) {
  const handle = new IOBase(path.join(root, name))
  handle.writeArrowBatchReader(
    new arrow.Table({ id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()) }),
  )

  // Identical calls on both sides, whatever the coding is.
  assert.equal(handle.readArrowBatchReader().toTable().numRows, 2, name)
  written.push(handle.readBytes())
}

// The bytes underneath are framed by the coding the name declared.
assert.deepEqual([...written[1].subarray(0, 2)], [0x1f, 0x8b])
assert.deepEqual([...written[2].subarray(0, 4)], [0x28, 0xb5, 0x2f, 0xfd])
assert.notDeepEqual(written[0], written[1])

fs.rmSync(root, { recursive: true, force: true })

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('@yggdryl/node')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.arrows.gz'))

const ids = Array.from({ length: 512 }, (_, index) => BigInt(index))
handle.writeArrowBatchReader(
  new arrow.Table({ id: arrow.vectorFromArray(ids, new arrow.Int64()) }),
  handle.recordOptions().withLevel(9),
)

assert.equal(handle.readArrowBatchReader().toTable().numRows, 512)
// Still a gzip member, and smaller than the stream it encodes.
assert.deepEqual([...handle.readBytes().subarray(0, 2)], [0x1f, 0x8b])
assert.ok(handle.size < 512 * 8)

fs.rmSync(root, { recursive: true, force: true })

## Options

In [ ]:
const assert = require('node:assert/strict')
const { Field, RecordOptions, fields } = require('@yggdryl/node')

const schema = fields.struct('row', [Field.from('id: int64')], { nullable: false })

// The media type names the encoding, so there is no format argument.
const options = new RecordOptions('trades.arrows')
options.schema = schema
options.level = 9

assert.ok(options.schema.equals(schema))
assert.equal(options.rootName, 'row')
assert.equal(options.level, 9)

options.batchSize = 1024
assert.equal(options.batchSize, 1024)

assert.equal(options.mimeType.toString(), 'application/vnd.apache.arrow.stream')
// `with*` returns a new value rather than changing the one it was built from.
assert.equal(options.withSafe(true).safe, true)
assert.equal(options.safe, false)

## Absence

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('@yggdryl/node')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))

// A resource that does not exist yet holds no batches; it is not a parse failure.
const missing = new IOBase(path.join(root, 'missing.arrows'))
assert.ok(!missing.exists())
assert.equal(missing.readArrowBatchReader().toTable().numRows, 0)

// Writing no batches still writes the schema, so the stream exists and reads.
const schema = new arrow.Schema([new arrow.Field('id', new arrow.Int64(), true)])
const written = new IOBase(path.join(root, 'empty.arrows'))
written.writeArrowBatchReader(new arrow.Table(schema))
assert.ok(written.size > 0)
assert.equal(written.readArrowBatchReader().toTable().numRows, 0)
assert.equal(written.readArrowField().name, 'row')

fs.rmSync(root, { recursive: true, force: true })

In [ ]:
const assert = require('node:assert/strict')
const { IOBase, MimeType } = require('@yggdryl/node')

const handle = IOBase.fromBytes(Buffer.from('definitely not an Arrow IPC stream'))
handle.mediaType = MimeType.ARROW_STREAM

assert.throws(() => handle.readArrowField(), /Arrow/)
assert.throws(() => handle.readArrowBatchReader(), /Arrow/)